# Complexity analysis - validation using the Potts model

**Aim:**
To evaluate four complexity metrics on model data:
1. Entropy rate
2. Excess entropy
3. Lempel-Ziv complexity
4. Hurst exponent (persistence) via Detrended Fluctuation Analysis (DFA)

**Note:** This script analyzes **_continuous microstate sequences_**.

**Procedure:**
1. Use data generated by the 2D Potts model with $Q$ states across a range of temperatures that contains the model's phase transition. 
2. Evaluate single-pixel time courses as if they were microstate sequences over $Q$ classes.
3. Visualize the metrics across the full temperature range including the phase transition.

**Data**: Simulated Potts system data.

## Model parameters

In [ ]:
# JupyterLite/Pyodide only: install the pure-Python mstsa build (no numba/C
# extensions -- see https://github.com/Frederic-vW/mstsa/tree/pyodide). Falls
# through silently on a normal Jupyter install, where mstsa is already present.
try:
    import piplite
    await piplite.install(
        "https://raw.githubusercontent.com/Frederic-vW/mstsa/pyodide/wheels/mstsa-0.4.3-py3-none-any.whl"
    )
except ImportError:
    pass


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import mstsa

data_path = "data/potts"

Q = 5 # number of discrete states, corresponds to K microstate classes
Tc = 1 / np.log(1 + np.sqrt(Q)) # critical temperature
print(f"Potts 2D model (25x25 lattice) Q={Q:d}, Tc={Tc:.2f}")

# relative temperatures (T / Tc)
rtemps = [0.2, 0.4, 0.6, 0.8, 0.9, 1.0, 1.1, 1.2, 1.4, 1.6, 1.8, 2.0, 2.2,
          2.4, 2.6, 2.8, 3.0]
n_temps = len(rtemps)
n_samples = 10  # lattice sites per file

## Sample timecourses

- Visualize one example Potts model time course
- Sample one lattice site below / at / above the critical temperature $T_c$

In [ ]:
tmax = 5000  # number of samples, cut-off for visibility
rtemp0, rtemp1, rtemp2 = 0.8, 1.0, 2.0
x0 = np.load(f"{data_path}/PottsQ5_Temp_{rtemp0 * Tc:.2f}_Lattice_L25_fm.npy")[:tmax, 3]
x1 = np.load(f"{data_path}/PottsQ5_Temp_{rtemp1 * Tc:.2f}_Lattice_L25_fm.npy")[:tmax, 6]
x2 = np.load(f"{data_path}/PottsQ5_Temp_{rtemp2 * Tc:.2f}_Lattice_L25_fm.npy")[:tmax, 0]

fig, ax = plt.subplots(3, 1, figsize=(15, 9))
for a, x, rtemp in zip(ax, [x0, x1, x2], [rtemp0, rtemp1, rtemp2]):
    a.plot(x, '-k')
    a.set_ylabel("state q", fontsize=18)
    a.set_title(f"rel. T: {rtemp:.1f}; T = {rtemp * Tc:.2f}", fontsize=18)
ax[-1].set_xlabel("time (samples)", fontsize=18)
plt.tight_layout()
os.makedirs('figures', exist_ok=True)
plt.savefig('figures/04_sample_timecourses.png', dpi=150, bbox_inches='tight')
plt.show()

## Complexity metrics

- **Entropy rate / excess entropy**: `mstsa.entropy_rate(x, Q, kmax)` fits joint entropy
  vs. block length over blocks `1..kmax+1` and returns `(er, ee)` (slope and
  intercept). In the current code version, `kmax=5` uses block lengths `1..kmax+1` = `1..6`).
- **LZ76**: `mstsa.lz76(x)`: normalized Lempel-Ziv complexity.
- **DFA**: `mstsa.dfa(x, lmin, lmax, fitmin, fitmax, nsteps)` computes the Hurst
  exponent from a random walk embedding. 
  _Caveat_: `mstsa.dfa` has no guard for constant input (as could happen with the Potts 
  model at low temperature); this can cause divide by zero and return NaN. Here, we just set `H=0.5` for a constant input sequence.

Per-temperature results are cached to disk (`data/cache/potts_complexity/*.npz`).

### How are entropy rate and excess entropy calculated?

- Short answer: it is surprisingly easy
- compute the block entropy for different block sizes $k$, $H(X_{t}, ..., X_{t+k-1})$
- fit a line to the $(k,H(k))$ data
- entropy rate: slope ; excess entropy: y-intercept

In [ ]:
# illustrate the entropy-rate / excess-entropy estimate: joint entropy H(k)
# vs. block length k is fit by a line whose slope is the entropy rate and
# whose intercept (at k=0) is the excess entropy. Block entropies are
# computed with mstsa.hk (same building block mstsa.entropy_rate uses
# internally), but the figure is drawn here so the fit line can be extended
# to k=0 and the intercept read off directly. Shown for one time course each
# at the critical temperature T_c and at a high temperature, to contrast a
# structured vs. a near-random (high-entropy-rate, low-excess-entropy) regime.
kmax_demo = 5
ks_demo = np.arange(1, kmax_demo + 2)
site_demo = 6


def plot_block_entropy(ax, rtemp, site=site_demo, kmax=kmax_demo, ks=ks_demo):
    x = np.load(f"{data_path}/PottsQ5_Temp_{rtemp * Tc:.2f}_Lattice_L25_fm.npy")[:, site].astype(np.int32)
    h = np.array([mstsa.hk(x, Q, k) for k in ks])
    er, ee = np.polyfit(ks, h, 1)

    ax.plot(ks, h, 'ok', ms=12, alpha=0.6, label='joint entropy $H(k)$', zorder=3)
    k_fit = np.array([0, ks[-1]])
    ax.plot(k_fit, er * k_fit + ee, '-b', label='fit', zorder=2)
    ax.plot(0, ee, 'sb', ms=10, zorder=4)
    # place the label in the empty upper-left corner (small k, large H) and
    # let the arrow run down along the y-axis to the intercept, rather than
    # crossing the ascending data/fit line
    ax.annotate(f"excess entropy\n$E$ = {ee:.3f} bit", xy=(0, ee), xycoords='data',
                xytext=(0.08, 0.92), textcoords='axes fraction',
                ha='left', va='top', fontsize=11, color='b',
                arrowprops=dict(arrowstyle='->', color='b'))
    ax.axvline(0, color='gray', lw=0.8, ls=':')
    ax.set_xlim(-0.3, ks[-1] + 0.3)
    ax.set_xlabel("block length k", fontsize=14)
    ax.set_title(f"rel. T = {rtemp:.1f}\n" + r"$h_X$" + f" = {er:.3f} bit/sample", fontsize=14)
    ax.grid(True, alpha=0.3)
    return er, ee


fig, axes = plt.subplots(1, 2, figsize=(11, 5))
er_tc, ee_tc = plot_block_entropy(axes[0], rtemp=1.0)
axes[0].set_ylabel(r"joint entropy $H\left(\mathbf{X}_n^{(k)}\right)$", fontsize=14)
axes[0].legend(fontsize=11, loc='lower right')

er_hi, ee_hi = plot_block_entropy(axes[1], rtemp=3.0)

plt.tight_layout()
os.makedirs('figures', exist_ok=True)
plt.savefig('figures/04_block_entropy_example.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"T_c  (rel. T=1.0): entropy rate = {er_tc:.3f} bit/sample, excess entropy = {ee_tc:.3f} bit")
print(f"high (rel. T=3.0): entropy rate = {er_hi:.3f} bit/sample, excess entropy = {ee_hi:.3f} bit")


**Observations:**
- around the critical temperature (rel. temp. $1.0$): medium entropy rate (randomness), large statistical complexity (excess entropy)
- far above the critical temperature (rel. temp. $3.0$): very high entropy rate (randomness), very low statistical complexity (excess entropy)

In [ ]:
k_hist = 6  # samples; mstsa kmax = k_hist - 1 (see markdown above)
kmax = k_hist - 1

p_dfa = dict(lmin=50, lmax=2500, fitmin=50, fitmax=2500, nsteps=50)

cache_dir = "data/cache/potts_complexity"
os.makedirs(cache_dir, exist_ok=True)

_KEYS = ['er', 'ee', 'lzc', 'h']


def safe_dfa(x, **kwargs):
    """mstsa.dfa on a constant sequence divides by zero (zero fluctuation at every
    scale) and returns NaN; the reference implementation instead returns H=0.5."""
    if np.all(x == x[0]):
        return 0.5
    return mstsa.dfa(x, **kwargs)


def _compute_temp(f_in, Q, kmax, p_dfa, n_samples):
    x = np.load(f_in).astype(np.int32)
    er = np.zeros(n_samples)
    ee = np.zeros(n_samples)
    lzc = np.zeros(n_samples)
    h = np.zeros(n_samples)
    for j in range(n_samples):
        xj = x[:, j]
        er[j], ee[j] = mstsa.entropy_rate(xj, Q, kmax=kmax)
        lzc[j] = mstsa.lz76(xj)
        h[j] = safe_dfa(xj.astype(np.float64), **p_dfa)
    return dict(er=er, ee=ee, lzc=lzc, h=h)


def analyze_temp(rtemp, Q=Q, Tc=Tc, kmax=kmax, p_dfa=p_dfa, n_samples=n_samples,
                  cache_dir=cache_dir, force=False):
    temp = rtemp * Tc
    f_in = f"{data_path}/PottsQ5_Temp_{temp:.2f}_Lattice_L25_fm.npy"
    cache_file = os.path.join(cache_dir, f"rtemp_{rtemp:.2f}.npz")
    if os.path.exists(cache_file) and not force:
        d = np.load(cache_file)
        if all(k in d.files for k in _KEYS):
            return {k: d[k] for k in _KEYS}
    res = _compute_temp(f_in, Q, kmax, p_dfa, n_samples)
    np.savez(cache_file, **res)
    return res


er_arr = np.zeros((n_temps, n_samples))
ee_arr = np.zeros((n_temps, n_samples))
lzc_arr = np.zeros((n_temps, n_samples))
h_arr = np.zeros((n_temps, n_samples))

for i, rtemp in enumerate(rtemps):
    print(f"[{i + 1:2d}/{n_temps}] rel. temp: {rtemp:.1f}, temp.: {rtemp * Tc:.2f}")
    res = analyze_temp(rtemp)
    er_arr[i] = res['er']
    ee_arr[i] = res['ee']
    lzc_arr[i] = res['lzc']
    h_arr[i] = res['h']
print("done.")

## Complexity measures vs. relative temperature

In [ ]:
fsize = 14
fig, ax = plt.subplots(3, 1, figsize=(9, 9))

# entropy rate + excess entropy
ax[0].plot(rtemps, er_arr.mean(axis=1), '-sk')
ax[0].set_ylabel("entropy rate (bits/sample)", fontsize=fsize)
ax0c = ax[0].twinx()
ax0c.set_ylabel("excess entropy (bits)", color='b', fontsize=fsize)
ax0c.plot(rtemps, ee_arr.mean(axis=1), '-^b')

# LZC vs ER
ax[1].plot(rtemps, lzc_arr.mean(axis=1), '-sk', label='LZC')
ax[1].plot(rtemps, er_arr.mean(axis=1), 'og', mfc='none', ms=14, label='ER')
ax[1].set_ylabel("LZC (bits/sample)", fontsize=fsize)
ax[1].legend(loc='lower right', fontsize=fsize)

# Hurst exponent: symbolic DFA
ax[2].plot(rtemps, h_arr.mean(axis=1), '-sk', label='DFA (symbolic)')
ax[2].set_ylabel("H", fontsize=fsize)
ax[2].set_xlabel("relative T (T/Tc)", fontsize=fsize)
ax[2].legend(loc='upper right', fontsize=fsize)

for a in ax:
    a.axvline(1.0, color='gray', ls=':', lw=1)

plt.tight_layout()
os.makedirs('figures', exist_ok=True)
plt.savefig('figures/04_complexity_vs_temp.png', dpi=150, bbox_inches='tight')
plt.show()

## Conclusions

1. Entropy rate and Lempel-Ziv complexity (LZC) behave like type I complexity
2. Excess entropy and Hurst exponent behave like type II complexity
3. Entropy rate and LZC measure the same quantity (large $n$ needed, they have opposite low-$n$ bias); also see Amigó et al.

**References:**
- von Wegner, F., Wiemers, M., Hermann, G., Tödt, I., Tagliazucchi, E., & Laufs, H. (2023). Complexity measures for EEG microstate sequences: concepts and algorithms. *Brain Topography*, 37, 296-311.
- Wu, F. Y. (1982). The Potts model. *Reviews of Modern Physics*, 54(1), 235-268.
- Amigó, J. M., Szczepański, J., Wajnryb, E., et al. (2004). Estimating the entropy rate of spike trains via Lempel-Ziv complexity. *Neural Computation*, 16, 717-736. https://doi.org/10.1162/089976604322860677
- Sleigh, J., & Hight, D. (2021). Is complexity complicated? *British Journal of Anaesthesia*, 127, 173-174. https://doi.org/10.1016/j.bja.2021.05.014